<a href="https://colab.research.google.com/github/rounak393/PRISM-Net-Parallel-ResNet-Integrated-SAM-with-Multi-attention-for-Breast-Segmentation/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install grad-cam

import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import albumentations as A

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 8
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

# ==========================================
# 1. DATASET
# ==========================================
class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform

        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir):
                continue

            images = sorted([
                f for f in os.listdir(cls_dir)
                if f.endswith(".png") and "_mask" not in f
            ])

            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")

                mask_files = sorted([
                    f for f in os.listdir(cls_dir)
                    if f.startswith(base_name + "_mask") and f.endswith(".png")
                ])

                if len(mask_files) == 0:
                    continue

                mask_paths = [os.path.join(cls_dir, f) for f in mask_files]
                self.samples.append((img_path, mask_paths))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            mask = (mask > 0).astype(np.uint8)
            combined_mask = np.logical_or(combined_mask, mask)

        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image = augmented["image"]
            combined_mask = augmented["mask"]

        combined_mask = (combined_mask > 0.5).astype(np.float32)
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(combined_mask).unsqueeze(0).float()

        return image, mask

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
    A.GridDistortion(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

# ==========================================
# 2. MODULES & ENCODERS
# ==========================================
class FastSAMEncoder(nn.Module):
    _IN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    _IN_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def __init__(self, checkpoint='FastSAM-x.pt', out_channels=512, fine_tune_last=2, feature_size=4, image_size=256):
        super().__init__()
        self.out_ch = out_channels
        self.feature_size = feature_size
        self.fine_tune_last = fine_tune_last

        from ultralytics import FastSAM as _FastSAM
        fs = _FastSAM(checkpoint)
        all_layers = list(fs.model.model)
        del fs

        target_h = image_size // 8
        selected = []
        backbone_out_ch = None
        probe = torch.zeros(1, 3, image_size, image_size)

        with torch.no_grad():
            x = probe
            for layer in all_layers:
                x = layer(x)
                selected.append(layer)
                if x.shape[-2] == target_h:
                    backbone_out_ch = x.shape[1]
                    break

        self.backbone = nn.ModuleList(selected)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for layer in self.backbone[-fine_tune_last:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.proj = nn.Sequential(
            nn.Conv2d(backbone_out_ch, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.register_buffer('in_mean', self._IN_MEAN)
        self.register_buffer('in_std', self._IN_STD)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.in_std + self.in_mean
        frozen_end = len(self.backbone) - self.fine_tune_last
        with torch.no_grad():
            for layer in self.backbone[:frozen_end]:
                x = layer(x)
        for layer in self.backbone[frozen_end:]:
            x = layer(x)

        x = self.proj(x)
        if x.shape[-1] != self.feature_size:
            x = F.interpolate(x, size=(self.feature_size, self.feature_size), mode='bilinear', align_corners=False)
        return x

class RobertsEdgeOperator(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[[[1.0, 0.0], [0.0, -1.0]]]])
        ky = torch.tensor([[[[0.0, 1.0], [-1.0, 0.0]]]])
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x_unnorm = x * self.std + self.mean
        gray = (0.2989 * x_unnorm[:, 0:1] + 0.5870 * x_unnorm[:, 1:2] + 0.1140 * x_unnorm[:, 2:3])
        gray_padded = F.pad(gray, (0, 1, 0, 1), mode="replicate")
        gx   = F.conv2d(gray_padded, self.kx)
        gy   = F.conv2d(gray_padded, self.ky)
        edge = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)
        return edge

class PCBAM_Filter(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.cam = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels)
        )
        self.sam = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.cam(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mxp = self.cam(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(avg + mxp)

        sp  = torch.cat([torch.mean(x_c, dim=1, keepdim=True), torch.max(x_c, dim=1, keepdim=True)[0]], dim=1)
        x_s = x_c * torch.sigmoid(self.sam(sp))
        return x_s

class SpatialGateAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid = in_channels // 8
        self.q_conv   = nn.Conv2d(in_channels, mid, 1)
        self.k_conv   = nn.Conv2d(in_channels, mid, 1)
        self.v_conv   = nn.Conv2d(in_channels, in_channels, 1)
        self.gate     = nn.Conv2d(mid * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q = self.q_conv(x)
        k = self.k_conv(x)
        v = self.v_conv(x)
        A = torch.sigmoid(self.gate(torch.cat([q, k, v], dim=1)))
        return x + v * A

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[-2:] != x1.shape[-2:]:
            g1 = F.interpolate(g1, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class PreTrainedDualPathUNet(nn.Module):
    def __init__(self, fastsam_checkpoint='FastSAM-x.pt'):
        super().__init__()
        self.global_encoder = FastSAMEncoder(checkpoint=fastsam_checkpoint, out_channels=512, fine_tune_last=2, feature_size=IMG_SIZE // 64, image_size=IMG_SIZE)
        self.roberts = RobertsEdgeOperator()

        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        old_conv = resnet.conv1
        self.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = old_conv.weight
            self.conv1.weight[:, 3]  = old_conv.weight.mean(dim=1)

        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.pcbam1 = PCBAM_Filter(256)
        self.pcbam2 = PCBAM_Filter(512)
        self.pcbam3 = PCBAM_Filter(1024)
        self.pcbam4 = PCBAM_Filter(2048)

        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(2048, 1024)
        self.fusion = DoubleConv(1024 + 512, 1024)
        self.sga    = SpatialGateAttention(1024)

        self.ag4 = AttentionGate(F_g=512, F_l=2048, F_int=512)
        self.ag3 = AttentionGate(F_g=256, F_l=1024, F_int=256)
        self.ag2 = AttentionGate(F_g=128, F_l=512,  F_int=128)
        self.ag1 = AttentionGate(F_g=64,  F_l=256,  F_int=64)

        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(512 + 2048, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(256 + 1024, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(128 + 512, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(64 + 256, 64)
        self.up0  = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = DoubleConv(64 + 64, 64)

        self.up_out  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final   = nn.Conv2d(32, 1, 1)

        self.out_ds2 = nn.Conv2d(128, 1, kernel_size=1)
        self.out_ds1 = nn.Conv2d(64,  1, kernel_size=1)

    def forward(self, x):
        global_feat = self.global_encoder(x)
        x_edge  = self.roberts(x)
        x_fused = torch.cat([x, x_edge], dim=1)

        x0 = self.relu(self.bn1(self.conv1(x_fused)))
        x1 = self.maxpool(x0)

        e1 = self.layer1(x1)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        s1 = self.pcbam1(e1)
        s2 = self.pcbam2(e2)
        s3 = self.pcbam3(e3)
        s4 = self.pcbam4(e4)

        b = self.bottleneck(self.pool(e4))

        if global_feat.shape[-2:] != b.shape[-2:]:
            global_feat = F.interpolate(global_feat, size=b.shape[-2:], mode='bilinear', align_corners=False)

        b = self.fusion(torch.cat([b, global_feat], dim=1))
        b = self.sga(b)

        g4 = self.up4(b)
        x4 = self.ag4(g=g4, x=s4)
        d4 = self.dec4(torch.cat([g4, x4], dim=1))

        g3 = self.up3(d4)
        x3 = self.ag3(g=g3, x=s3)
        d3 = self.dec3(torch.cat([g3, x3], dim=1))

        g2 = self.up2(d3)
        x2 = self.ag2(g=g2, x=s2)
        d2 = self.dec2(torch.cat([g2, x2], dim=1))

        out_ds2 = self.out_ds2(d2)

        g1 = self.up1(d2)
        x1 = self.ag1(g=g1, x=s1)
        d1 = self.dec1(torch.cat([g1, x1], dim=1))

        out_ds1 = self.out_ds1(d1)

        d0  = self.dec0(torch.cat([self.up0(d1), x0], dim=1))
        out = self.dec_out(self.up_out(d0))

        out_main = self.final(out)
        if self.training:
            return out_main, out_ds1, out_ds2
        else:
            return out_main

# ==========================================
# 3. METRICS & LOSS
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self, smooth=1.0, focal_gamma=2.0, alpha=0.3, beta=0.7):
        super().__init__()
        self.smooth = smooth
        self.focal_gamma = focal_gamma
        self.alpha = alpha
        self.beta = beta

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        tgt = targets.view(-1)

        tp = (probs * tgt).sum()
        fp = (probs * (1.0 - tgt)).sum()
        fn = ((1.0 - probs) * tgt).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        tversky_loss = 1.0 - tversky

        pt = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma * torch.log(pt + 1e-8)).mean()

        return 0.4 * bce + 0.4 * tversky_loss + 0.2 * focal

def hard_dice_coef(y_true, y_pred_probs, threshold=0.6, smooth=1e-5):
    y_pred_bin = (y_pred_probs > threshold).float()
    y_true_f = y_true.view(-1)
    y_pred_f = y_pred_bin.view(-1)
    inter = (y_true_f * y_pred_f).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

def hard_iou_score(y_pred_probs, masks, threshold=0.6, eps=1e-6):
    preds_bin = (y_pred_probs > threshold).float()
    masks_bin = (masks > threshold).float()
    inter = (preds_bin * masks_bin).sum().item()
    union = preds_bin.sum().item() + masks_bin.sum().item() - inter
    return inter / (union + eps)

# ==========================================
# 4. TRAINING & EVALUATION LOOP
# ==========================================
def train_model(model, train_loader, val_loader, epochs=50, threshold=0.6):
    criterion = HybridLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_model_weights = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_dice = 0

        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, masks = images.to(device), masks.to(device)
            preds_logits, logits_ds1, logits_ds2 = model(images)

            mask_ds1 = F.interpolate(masks, size=logits_ds1.shape[-2:], mode='nearest-exact')
            mask_ds2 = F.interpolate(masks, size=logits_ds2.shape[-2:], mode='nearest-exact')

            loss = criterion(preds_logits, masks) + 0.3 * criterion(logits_ds1, mask_ds1) + 0.1 * criterion(logits_ds2, mask_ds2)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            preds_probs = torch.sigmoid(preds_logits)
            train_dice += hard_dice_coef(masks, preds_probs, threshold=threshold).item()

        model.eval()
        val_loss = 0
        val_dice = 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, masks = images.to(device), masks.to(device)
                preds_logits = model(images)

                loss = criterion(preds_logits, masks)
                val_loss += loss.item()

                preds_probs = torch.sigmoid(preds_logits)
                val_dice += hard_dice_coef(masks, preds_probs, threshold=threshold).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_train_dice = train_dice / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)
        avg_val_dice   = val_dice / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"\nEpoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Train Dice: {avg_train_dice:.4f} | Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "best_fastsam_dualpath_unet.pth")
            print(f"✓ Best model saved (Min Val Loss: {best_val_loss:.4f})")

        scheduler.step(avg_val_loss)

    model.load_state_dict(best_model_weights)
    return model

# ==========================================
# 5. ERROR ANALYSIS (TP/FP/FN MAPPING)
# ==========================================
def evaluate_model(model, test_loader, threshold=0.6):
    model.eval()
    test_dice, test_iou = 0, 0
    mean_std = (torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225]))

    vis_images, vis_true, vis_pred = None, None, None

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Testing"):
            images, masks = images.to(device), masks.to(device)
            preds_probs = torch.sigmoid(model(images))

            test_dice += hard_dice_coef(masks, preds_probs, threshold=threshold).item()
            test_iou += hard_iou_score(preds_probs, masks, threshold=threshold)

            if vis_images is None:
                vis_images = images.cpu()
                vis_true = masks.cpu()
                vis_pred = (preds_probs > threshold).float().cpu()

    print(f"\nFinal Test Dice: {test_dice / len(test_loader):.4f}")
    print(f"Final Test IoU:  {test_iou / len(test_loader):.4f}")

    for i in range(min(3, vis_images.size(0))):
        img_unnorm = vis_images[i].permute(1, 2, 0) * mean_std[1] + mean_std[0]
        img_unnorm = np.clip(img_unnorm.numpy(), 0, 1)

        t_mask = vis_true[i, 0].numpy()
        p_mask = vis_pred[i, 0].numpy()

        error_map = np.zeros((*t_mask.shape, 3), dtype=np.uint8)
        error_map[(p_mask == 1) & (t_mask == 1)] = [255, 255, 255] # White: TP
        error_map[(p_mask == 1) & (t_mask == 0)] = [255, 0, 0]     # Red: FP
        error_map[(p_mask == 0) & (t_mask == 1)] = [0, 0, 255]     # Blue: FN

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_unnorm)
        axes[0].set_title("Input Image")
        axes[1].imshow(t_mask, cmap='gray')
        axes[1].set_title("Ground Truth")
        axes[2].imshow(error_map)
        axes[2].set_title("Error Map (W:TP, R:FP, B:FN)")

        for ax in axes: ax.axis('off')
        plt.show()

# ==========================================
# 6. GRAD-CAM
# ==========================================
class SemanticSegmentationTarget:
    def __init__(self, mask): self.mask = mask
    def __call__(self, model_output): return (model_output * self.mask).sum()

def generate_grad_cam(model, image_tensor, target_mask):
    model.eval()
    target_layers = [model.dec_out]
    cam = GradCAM(model=model, target_layers=target_layers)
    targets = [SemanticSegmentationTarget(target_mask.to(device))]

    grayscale_cam = cam(input_tensor=image_tensor.to(device), targets=targets)[0, :]

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
    rgb_img = (image_tensor[0].to(device) * std + mean).permute(1, 2, 0).cpu().numpy()
    rgb_img = np.clip(rgb_img, 0, 1)

    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb_img); axes[0].set_title("Input"); axes[0].axis('off')
    axes[1].imshow(target_mask[0, 0].cpu().numpy(), cmap='gray'); axes[1].set_title("Target Mask"); axes[1].axis('off')
    axes[2].imshow(cam_image); axes[2].set_title("Grad-CAM Overlay"); axes[2].axis('off')
    plt.show()

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
    indices = list(range(len(full_dataset)))
    np.random.shuffle(indices)

    train_size = int(0.8 * len(full_dataset))
    val_size   = int(0.1 * len(full_dataset))

    train_dataset = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform), indices[val_size:train_size + val_size])
    val_dataset   = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[:val_size])
    test_dataset  = torch.utils.data.Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[train_size + val_size:])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = PreTrainedDualPathUNet(fastsam_checkpoint='FastSAM-x.pt').to(device)

    # 1. Train model (Strictly tracks and logs 4 metrics)
    model = train_model(model, train_loader, val_loader, epochs=100, threshold=0.6)

    # 2. Load the best model weights based strictly on minimum validation loss
    model.load_state_dict(torch.load("best_fastsam_dualpath_unet.pth", map_location=device))

    # 3. Test on test dataset
    evaluate_model(model, test_loader, threshold=0.6)

    # 4. Generate Grad-CAM sample
    sample_img, sample_mask = next(iter(test_loader))
    generate_grad_cam(model, sample_img[0:1], sample_mask[0:1])